In [16]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical
import sys
sys.path.append('..')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random
import numpy as np
import random
from tqdm import trange
import matplotlib as plt
import torch.optim as optim
import gymnasium as gym
from gymnasium.vector import SyncVectorEnv
import matplotlib.pyplot as plt
from burauEnv import BurauEnv
MOD = 2
OBS_SPACE = 32

# ---------- Hyper‑parameters ----------
ENV_ID          = "CartPole-v1"
NUM_ENVS        = 8              # parallel environments
ROLLOUT_STEPS   = 5              # T in the notes (how often we update)
TOTAL_UPDATES   = 2_000          # training iterations
GAMMA           = 0.99           # discount factor
ENTROPY_BETA    = 0.01           # exploration bonus
CRITIC_COEF     = 0.5            # scales value loss
LR              = 2e-4           # Adam learning rate
GRAD_CLIP       = 0.5            # global‑norm clipping
PRINT_EVERY     = 100            # how often to log progress
# --------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Make a vector of BurauEnvs ─────────────────────────────────────────────────
def make_env(mod=MOD, obs=OBS_SPACE):
    def _init():
        return BurauEnv(mod=mod, obs=obs)
    return _init

venv = SyncVectorEnv([make_env() for _ in range(NUM_ENVS)])

envs = gym.vector.SyncVectorEnv([make_env() for _ in range(NUM_ENVS)])
obs, _ = envs.reset(seed=0)
obs_shape = obs.shape[1:]          # (4,) for CartPole
act_dim  = envs.single_action_space.n

# ---------- Network ----------
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh()
        )
        self.policy_head = nn.Linear(128, act_dim)
        self.value_head  = nn.Linear(128, 1)

    def forward(self, x):
        x = self.backbone(x)
        return self.policy_head(x), self.value_head(x)

net = ActorCritic(np.prod(obs_shape), act_dim).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=LR)

# ---------- Storage helpers ----------
obs_buf      = np.zeros((ROLLOUT_STEPS, NUM_ENVS) + obs_shape, dtype=np.float32)
acts_buf     = np.zeros((ROLLOUT_STEPS, NUM_ENVS),               dtype=np.int32)
rews_buf     = np.zeros((ROLLOUT_STEPS, NUM_ENVS),               dtype=np.float32)
dones_buf    = np.zeros((ROLLOUT_STEPS, NUM_ENVS),               dtype=bool)
logp_buf     = np.zeros((ROLLOUT_STEPS, NUM_ENVS),               dtype=np.float32)
values_buf   = np.zeros((ROLLOUT_STEPS, NUM_ENVS),               dtype=np.float32)

# ---------- Training loop ----------
global_step = 0
for update in range(1, TOTAL_UPDATES + 1):
    for step in range(ROLLOUT_STEPS):
        obs_buf[step] = obs
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
        logits, value = net(obs_t)
        dist = Categorical(logits=logits)
        action = dist.sample()

        next_obs, reward, terminated, truncated, _ = envs.step(action.cpu().numpy())
        done = np.logical_or(terminated, truncated)

        # Store
        acts_buf[step]   = action.cpu().numpy()
        rews_buf[step]   = reward
        dones_buf[step]  = done
        logp_buf[step] = dist.log_prob(action).detach().cpu().numpy()
        values_buf[step] = value.squeeze(-1).detach().cpu().numpy()

        obs = next_obs
        global_step += NUM_ENVS

    # Bootstrap V(s') for the last state in the buffer
    with torch.no_grad():
        next_obs_t = torch.tensor(obs, dtype=torch.float32, device=device)
        _, next_value = net(next_obs_t)
        next_value = next_value.squeeze(-1).cpu().numpy()

    # ---------- Compute advantages & targets (n = 1) ----------
    advantages = np.zeros_like(rews_buf)
    returns    = np.zeros_like(rews_buf)

    # TD(0): δ_t = r_t + γ V(s_{t+1}) – V(s_t)
    for step in reversed(range(ROLLOUT_STEPS)):
        if step == ROLLOUT_STEPS - 1:
            next_val = next_value
            next_done = dones_buf[step]
        else:
            next_val  = values_buf[step + 1]
            next_done = dones_buf[step + 1]

        td_target   = rews_buf[step] + GAMMA * next_val * (1.0 - next_done)
        advantages[step] = td_target - values_buf[step]
        returns[step]    = td_target          # for critic loss

    # Flatten rollout dimension
    obs_b   = torch.tensor(obs_buf.reshape(-1, *obs_shape), dtype=torch.float32, device=device)
    acts_b  = torch.tensor(acts_buf.flatten(),               dtype=torch.int64,  device=device)
    logp_b  = torch.tensor(logp_buf.flatten(),               dtype=torch.float32, device=device)
    adv_b   = torch.tensor(advantages.flatten(),             dtype=torch.float32, device=device)
    ret_b   = torch.tensor(returns.flatten(),                dtype=torch.float32, device=device)

    # Advantage normalisation (helps learning rate stability)
    adv_b = (adv_b - adv_b.mean()) / (adv_b.std() + 1e-8)

    # ---------- Losses ----------
    logits_b, val_b = net(obs_b)
    dist_b = Categorical(logits=logits_b)
    entropy = dist_b.entropy().mean()

    new_logp = dist_b.log_prob(acts_b)
    actor_loss  = -(new_logp * adv_b).mean()
    critic_loss = 0.5 * (ret_b - val_b.squeeze(-1)).pow(2).mean()
    loss = actor_loss + CRITIC_COEF * critic_loss - ENTROPY_BETA * entropy

    # ---------- Optimise ----------
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
    optimizer.step()

    # ---------- Logging ----------
    if update % PRINT_EVERY == 0:
        avg_return = returns.mean()
        print(f"Update {update:4d} | Loss {loss.item():.3f} | "
              f"Actor {actor_loss.item():.3f} | Critic {critic_loss.item():.3f} | "
              f"Ent {entropy.item():.3f} | Avg return {avg_return:.1f}")
        env = BurauEnv(2,OBS_SPACE)
        state, _ = env.reset()
        done = False
        while not done:
            with torch.no_grad():
                q_vals = net(torch.tensor(state, dtype=torch.float32).to(device))[1]
                action = torch.argmax(q_vals).item()
                state, _, term, trunc, _ = env.step(action)
                done = term or trunc
        env.render()
        print(env.power_range)
        

envs.close()
print("Training finished ✅")

Update  100 | Loss 5.822 | Actor -0.044 | Critic 11.758 | Ent 1.322 | Avg return -5.3
[Turn 32] word = AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
32
Update  200 | Loss 0.025 | Actor -0.605 | Critic 1.268 | Ent 0.353 | Avg return -8.2
[Turn 32] word = AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
32
Update  300 | Loss 0.038 | Actor -0.108 | Critic 0.299 | Ent 0.334 | Avg return -11.1
[Turn 32] word = AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
32
Update  400 | Loss 0.091 | Actor -0.000 | Critic 0.188 | Ent 0.289 | Avg return -12.6
[Turn 32] word = AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
32
Update  500 | Loss -0.007 | Actor -0.207 | Critic 0.409 | Ent 0.388 | Avg return -11.6
[Turn 32] word = AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
32
Update  600 | Loss -0.096 | Actor -0.652 | Critic 1.119 | Ent 0.346 | Avg return -4.4
[Turn 32] word = AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
32


KeyboardInterrupt: 